# GTAN UID-disjoint — full training-label reveal for validation + CNN/LSTM hybrids

This notebook is an upper-bound ablation for the UID-disjoint graph setting. During GTAN training, all training nodes are eligible as known-label neighbours, but seed nodes in the current supervised batch are masked to avoid an own-label shortcut. During validation, validation nodes remain unknown while all training labels are available as neighbour features. CNN and LSTM baselines are not recomputed; only CNN+GTAN and LSTM+GTAN are trained from the extracted GTAN embeddings.

In [34]:
!pip install -q torch_geometric

In [35]:
import torch
TORCH=torch.__version__
!pip install -q pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}.html
# optional FAISS backend (set SIM_BACKEND='faiss'): 
# !pip install -q faiss-cpu

In [36]:
import torch_scatter, torch_sparse
print('GNN backend ready')

GNN backend ready


In [37]:
# ===== Config =====
import os, gc, math, time, copy, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric.utils import add_remaining_self_loops
from torch_geometric.loader import NeighborLoader
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
OUT = Path("/kaggle/working"); OUT.mkdir(parents=True, exist_ok=True)

# ===== MODES =====
# Upper-bound ablation:
#   Train batches: all train labels may be used by neighbour nodes, but the
#   current seed nodes are masked before the forward pass.
#   Validation: validation labels are unknown, all train labels are available.
FULL_TRAIN_LABEL_REVEAL = True

# FAST_GTAN_RUN changes compute-heavy graph settings for a quick diagnostic run.
# Set this to False for the slower, more complete 3-seed / larger-neighbour run.
FAST_GTAN_RUN = True

# OOF train embeddings avoid own-label shortcut when embeddings are used by CNN/LSTM,
# but they require 5 additional full graph inference passes. Keep False for speed.
USE_OOF_TRAIN_EMBEDDINGS = False
RUN_HYBRIDS = True

# If FULL_TRAIN_LABEL_REVEAL=False, this falls back to the fixed-reference mode.
USE_REFERENCE_NODES    = True
ADD_TIME_COUNT_FEATURES = True
ADD_SIM_REF_FRAUD       = False

SEED=42; FRAC_TRAIN=0.8; REF_FRAC=0.40; REF_SEED=123
IDENTITY_COLS=["uid","card1_addr1","card1_addr1_P_emaildomain","DeviceInfo"]
USE_SIMILARITY_EDGES = True
EDGE_PER_TRANS=12; K_SIM=15
DECAY_TAUS=(1.0,7.0,30.0)
USE_COSINE=True
SIM_BACKEND="gpu"; SIM_BATCH=1024
GTAN_HIDDEN=32; GTAN_HEADS=4; GTAN_LAYERS=2; GTAN_DROP=0.2
GTAN_EPOCHS=30; GTAN_LR=3e-4; GTAN_WD=1e-5; EARLY_STOP_PATIENCE=4
PRUNE_TOPK=True; PRUNE_K=15
USE_WEIGHTED_SAMPLING = True
USE_TEMPORAL_SAMPLING = False
GTAN_BATCH=4096; GTAN_NEIGH=(PRUNE_K+1,10); N_SEEDS=3

if FAST_GTAN_RUN:
    # The original (4096, 16, 10) sampler can expand to roughly 655k neighbour
    # candidates per batch. This setting cuts that by more than an order of magnitude.
    N_SEEDS = 1
    GTAN_EPOCHS = 12
    EARLY_STOP_PATIENCE = 3
    PRUNE_K = 8
    GTAN_NEIGH = (PRUNE_K + 1, 4)
    GTAN_BATCH = 2048
    USE_WEIGHTED_SAMPLING = False
    print("FAST_GTAN_RUN enabled:", {
        "N_SEEDS": N_SEEDS,
        "GTAN_EPOCHS": GTAN_EPOCHS,
        "GTAN_BATCH": GTAN_BATCH,
        "GTAN_NEIGH": GTAN_NEIGH,
        "PRUNE_K": PRUNE_K,
        "USE_WEIGHTED_SAMPLING": USE_WEIGHTED_SAMPLING,
        "USE_OOF_TRAIN_EMBEDDINGS": USE_OOF_TRAIN_EMBEDDINGS,
    })

# ===== CNN/LSTM hybrid settings =====
SPLIT_DATA_DIR = "/kaggle/input/datasets/bachhoviet/split-data-npy"
WINDOW=20; BATCH=1024; EPOCHS=30; LR=1e-3; WEIGHT_DECAY=2e-4
GRAD_CLIP=1.0; EARLY_STOP_PATIENCE=6
HIDDEN_DIM=128; NUM_LAYERS=2; DROPOUT=0.3
USE_POS_WEIGHT=False; USE_STATIC_TOWER=True
CNN_CHANNELS=128; N_RESBLOCKS=3; N_ATTN_LAYERS=2; N_HEADS=4; ATTN_DROPOUT=0.2
STATIC_HIDDEN=64; USE_CLS=True; USE_MAX_POOL=True

# Previously recorded baselines; this notebook intentionally does not rerun them.
CNN_BASELINE_UID_DISJOINT_AUC = 0.9012
LSTM_BASELINE_UID_DISJOINT_AUC = 0.8864

# For downstream embeddings, train rows are extracted out-of-fold so a row's
# own target label is not encoded into the embedding consumed by CNN/LSTM.
TRAIN_EMB_OOF_FOLDS = 5

torch.manual_seed(SEED); np.random.seed(SEED)


device: cuda


In [38]:
# ===== Load data (copy4 has features + identity + TransactionDT) =====
df = pd.read_parquet("/kaggle/input/datasets/bachhoviet/parquets/X_train_copy4.parquet")
df.index = df.index.astype(np.int64)
y_all = pd.read_parquet("/kaggle/input/datasets/bachhoviet/parquets/y_train.parquet")["isFraud"]
y_all = y_all.reindex(df.index).to_numpy().astype(np.int64)

BASE_FEAT = [c for c in ['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3', 'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4_FE', 'M6', 'M7', 'M8', 'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41', 'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89', 'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210', 'V215', 'V218', 'V220', 'V221', 'V223', 'V224', 'V226', 'V228', 'V229', 'V234', 'V235', 'V238', 'V240', 'V250', 'V252', 'V253', 'V257', 'V258', 'V260', 'V261', 'V264', 'V266', 'V267', 'V271', 'V274', 'V277', 'V281', 'V283', 'V284', 'V285', 'V286', 'V289', 'V291', 'V294', 'V296', 'V297', 'V301', 'V303', 'V305', 'V307', 'V309', 'V310', 'V314', 'V320', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_15_FE', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_28', 'id_29', 'id_31_FE', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo_FE', 'cents', 'dollars', 'addr1_FE', 'card1_FE', 'card2_FE', 'card3_FE', 'card1_addr1', 'card1_addr1_P_emaildomain', 'card1_addr1_FE', 'card1_addr1_P_emaildomain_FE', 'TransactionAmt_card1_mean', 'TransactionAmt_card1_std', 'TransactionAmt_card1_addr1_mean', 'TransactionAmt_card1_addr1_std', 'TransactionAmt_card1_addr1_P_emaildomain_mean', 'TransactionAmt_card1_addr1_P_emaildomain_std', 'D9_card1_mean', 'D9_card1_std', 'D9_card1_addr1_mean', 'D9_card1_addr1_std', 'D9_card1_addr1_P_emaildomain_mean', 'D9_card1_addr1_P_emaildomain_std', 'D11_card1_mean', 'D11_card1_std', 'D11_card1_addr1_mean', 'D11_card1_addr1_std', 'D11_card1_addr1_P_emaildomain_mean', 'D11_card1_addr1_P_emaildomain_std', 'is_december', 'is_holiday', 'uid_FE', 'delta_seconds_prev', 'uid_count_so_far', 'uid_prev_amt', 'uid_amt_diff_prev', 'uid_amt_ratio_prev', 'uid_amt_cummean', 'uid_amt_cummax', 'DT_hour_sin', 'DT_hour_cos', 'DT_day_week_sin', 'DT_day_week_cos', 'DT_day_month_sin', 'DT_day_month_cos', 'DT_week_month_sin', 'DT_week_month_cos'] if c in df.columns]

EXTRA_FEAT = []
if ADD_TIME_COUNT_FEATURES:
    # number of transactions in that day / week / month for that uid
    day  = (df["TransactionDT"].to_numpy() // 86400).astype(np.int64)
    tmp  = pd.DataFrame({"uid": df["uid"].to_numpy(), "day": day, "week": day//7, "month": day//30})
    df["uid_day_ct"]   = tmp.groupby(["uid","day"]).transform("size").to_numpy()
    df["uid_week_ct"]  = tmp.groupby(["uid","week"]).transform("size").to_numpy()
    df["uid_month_ct"] = tmp.groupby(["uid","month"]).transform("size").to_numpy()
    EXTRA_FEAT = ["uid_day_ct","uid_week_ct","uid_month_ct"]
    print("added time-count features:", EXTRA_FEAT)

FEAT = BASE_FEAT + EXTRA_FEAT
print("node features:", len(FEAT), "| rows:", len(df))
X_raw   = df[FEAT].apply(pd.to_numeric, errors="coerce").fillna(-1).to_numpy().astype(np.float32)
times   = df["TransactionDT"].to_numpy().astype(np.float64)
uids    = df["uid"].to_numpy()
id_vals = {c: df[c].to_numpy() for c in IDENTITY_COLS}
N = len(df)


added time-count features: ['uid_day_ct', 'uid_week_ct', 'uid_month_ct']
node features: 215 | rows: 590540


In [39]:
# ===== uid-disjoint split + label-availability protocol =====
uid_counts = df["uid"].value_counts(dropna=False)
uids_arr = uid_counts.index.to_numpy(); counts_arr = uid_counts.values.astype(np.int64)
rng = np.random.default_rng(SEED); perm = rng.permutation(len(uids_arr))
uids_shuf = uids_arr[perm]; counts_shuf = counts_arr[perm]
cut = int(np.searchsorted(np.cumsum(counts_shuf), int(FRAC_TRAIN*counts_shuf.sum()))) + 1
train_uids = set(uids_shuf[:cut].tolist()); val_uids = set(uids_shuf[cut:].tolist())

u = df["uid"].to_numpy()
train_mask = np.isin(u, list(train_uids)); val_mask = ~train_mask
train_idx = np.flatnonzero(train_mask)
val_idx = np.flatnonzero(val_mask)

if FULL_TRAIN_LABEL_REVEAL:
    label_known_mask = train_mask.copy()
    query_mask = train_mask.copy()
    ref_mask = train_mask.copy()
    protocol_name = "FULL_TRAIN_LABEL_REVEAL"
else:
    if USE_REFERENCE_NODES:
        tr_uid_arr = np.array(sorted(train_uids))
        ref_rng = np.random.default_rng(REF_SEED)
        n_ref = int(REF_FRAC*len(tr_uid_arr))
        ref_uids = set(ref_rng.permutation(tr_uid_arr)[:n_ref].tolist())
        np.save(OUT/"reference_uids.npy", np.array(sorted(ref_uids)))
        ref_mask = np.isin(u, list(ref_uids))
        query_mask = train_mask & ~ref_mask
        label_known_mask = ref_mask.copy()
        protocol_name = f"FIXED_REFERENCE_{REF_FRAC:.0%}_TRAIN_UIDS"
    else:
        ref_mask = np.zeros(N, dtype=bool)
        query_mask = train_mask.copy()
        label_known_mask = np.zeros(N, dtype=bool)
        protocol_name = "NO_REFERENCE_LABEL_INPUT"

label_known_idx = np.flatnonzero(label_known_mask)
ref_idx = np.flatnonzero(ref_mask)
query_idx = np.flatnonzero(query_mask)

print(f"MODE: {protocol_name} | time_count_feats={ADD_TIME_COUNT_FEATURES}")
print(f"train rows={train_mask.sum():,} (known_label={label_known_mask.sum():,} query={query_mask.sum():,}) | val rows={val_mask.sum():,}")
print(f"train UIDs={len(train_uids):,} | val UIDs={len(val_uids):,}")
print(f"fraud: query={y_all[query_idx].mean():.4f} val={y_all[val_idx].mean():.4f}" +
      (f" known={y_all[label_known_idx].mean():.4f}" if len(label_known_idx) else " (no known label input)"))


MODE: reference_nodes=False | time_count_feats=True
train rows=472,434 (ref=0 query=472,434) | val rows=118,106
fraud: query=0.0346 val=0.0365 (no reference reveal)


In [40]:
# ===== Scale (fit on TRAIN) + L2-normalize for cosine =====
sc = StandardScaler().fit(X_raw[train_mask])
X_scaled = sc.transform(X_raw).astype(np.float32)       # GTAN node features
norm = np.linalg.norm(X_scaled, axis=1, keepdims=True); norm[norm==0]=1.0
X_norm = (X_scaled / norm).astype(np.float32)           # for cosine similarity
print("X_scaled", X_scaled.shape, "| X_norm", X_norm.shape)


X_scaled (590540, 215) | X_norm (590540, 215)


In [41]:
# ===== kNN similarity edges: GPU batched cosine top-k (causal, cross-uid) =====
@torch.no_grad()
def knn_gpu(Xn_np, times, uids, k, dev, batch=1024):
    N = Xn_np.shape[0]
    Xn = torch.from_numpy(Xn_np).to(dev)
    t  = torch.tensor(times, device=dev)
    uu = torch.tensor(pd.factorize(uids)[0], device=dev)   # int codes for uid
    src_l, dst_l = [], []
    for s0 in range(0, N, batch):
        s1 = min(s0+batch, N)
        sims = Xn[s0:s1] @ Xn.T                              # (B,N) cosine (normalized)
        qt = t[s0:s1].unsqueeze(1); qu = uu[s0:s1].unsqueeze(1)
        valid = (t.unsqueeze(0) < qt) & (uu.unsqueeze(0) != qu)   # earlier time AND different uid
        sims = sims.masked_fill(~valid, float("-inf"))
        kk = min(k, N)
        topv, topi = torch.topk(sims, kk, dim=1)
        m = torch.isfinite(topv)
        rows = (torch.arange(s1-s0, device=dev).unsqueeze(1).expand(-1, kk)[m] + s0)
        cols = topi[m]
        dst_l.append(rows.cpu()); src_l.append(cols.cpu())   # src=similar PAST -> dst=current
        del sims, valid, topv, topi
    if device.type=="cuda": torch.cuda.empty_cache()
    if not src_l: z=np.array([],np.int64); return z,z
    return torch.cat(src_l).numpy(), torch.cat(dst_l).numpy()

def knn_faiss(Xn_np, k):   # cosine via inner product on L2-normalized (no causal filter -> over-query then filter)
    import faiss
    index = faiss.IndexFlatIP(Xn_np.shape[1]); index.add(Xn_np)
    _, I = index.search(Xn_np, k+1)   # +1 to drop self
    return I


In [42]:
# ===== Build combined edge_index + 11-dim edge_attr (identity + similarity, coalesced) =====
def _valid_id(v):
    v=np.asarray(v); ok=v!=-1
    if np.issubdtype(v.dtype, np.floating): ok &= ~np.isnan(v)
    return ok

def _chain_pairs(labels, t, ept, valid=None):
    n=len(labels); pos=np.arange(n); gl=np.asarray(labels); tv=np.asarray(t)
    if valid is not None: pos=pos[valid]; gl=gl[valid]; tv=tv[valid]
    if len(pos)==0: z=np.array([],np.int64); return z,z
    order=np.argsort(tv,kind="mergesort")
    g=pd.DataFrame({"node":pos[order],"g":gl[order]}); sl,dl=[],[]
    for _,grp in g.groupby("g",sort=False):
        idx=grp["node"].to_numpy(); Lg=len(idx)
        for j in range(1,ept):
            if j>=Lg: break
            sl.append(idx[:Lg-j]); dl.append(idx[j:])
    if sl: return np.concatenate(sl), np.concatenate(dl)
    z=np.array([],np.int64); return z,z

def prune_topk_per_dst(ei, ea, K):
    # Keep only the top-K strongest IN-edges per destination node (deterministic).
    # strength = cosine + relation_count + recency_7d ; self-loops always kept.
    dst = ei[1].numpy()
    cos = ea[:,10].numpy() if ea.shape[1] >= 11 else np.zeros(ea.shape[0], np.float32)
    strength = cos + ea[:,9].numpy() + ea[:,1].numpy()
    is_self = ea[:,8].numpy() > 0.5
    e = pd.DataFrame({"dst":dst, "str":strength, "i":np.arange(len(dst))})
    e["rank"] = e.groupby("dst", sort=False)["str"].rank(method="first", ascending=False)
    keep = (e["rank"].to_numpy() <= K) | is_self
    idx = e["i"].to_numpy()[keep]
    return ei[:, idx], ea[idx]


def build_edges():
    parts=[]   # (src,dst,rel_idx)  rel: 0 uid,1 c1a1,2 c1a1e,3 dev,4 sim
    for ridx,col in enumerate(IDENTITY_COLS):
        s,d=_chain_pairs(id_vals[col], times, EDGE_PER_TRANS, valid=_valid_id(id_vals[col]))
        if len(s): parts.append((s,d,ridx)); print(f"  {col}: {len(s):,} edges")
    if USE_SIMILARITY_EDGES:
        if SIM_BACKEND=="gpu":
            s,d = knn_gpu(X_norm, times, uids, K_SIM, device, batch=SIM_BATCH)
        else:
            I = knn_faiss(X_norm, K_SIM); rows=np.repeat(np.arange(N),K_SIM); cols=I[:,1:].reshape(-1)
            keep = times[cols] < times[rows]
            keep &= (pd.factorize(uids)[0][cols] != pd.factorize(uids)[0][rows])
            s,d = cols[keep], rows[keep]
        if len(s): parts.append((s,d,4)); print(f"  similarity(kNN k={K_SIM}): {len(s):,} edges")
    else:
        print("  similarity edges: DISABLED (internal-only mode)")
    src=np.concatenate([p[0] for p in parts]); dst=np.concatenate([p[1] for p in parts])
    rel=np.concatenate([np.full(len(p[0]),p[2],np.int8) for p in parts])
    # coalesce duplicate (src,dst): OR relation flags
    e=pd.DataFrame({"s":src,"d":dst})
    for j in range(5): e[f"r{j}"]=(rel==j).astype(np.float32)
    agg=e.groupby(["s","d"],sort=False).max().reset_index()
    s2=agg["s"].to_numpy(); d2=agg["d"].to_numpy()
    relflags=agg[[f"r{j}" for j in range(5)]].to_numpy(np.float32)
    relcount=relflags.sum(1,keepdims=True)
    dt=((times[d2]-times[s2]).astype(np.float32))/86400.0
    rec=np.stack([np.exp(-dt/float(tt)) for tt in DECAY_TAUS],axis=1)
    cols_list=[rec, relflags, np.zeros((len(s2),1),np.float32), relcount]   # ..., is_self=0, rel_count
    if USE_COSINE:
        cos=np.empty((len(s2),1),np.float32); B=1_000_000
        for i in range(0,len(s2),B):
            cos[i:i+B,0]=(X_norm[s2[i:i+B]]*X_norm[d2[i:i+B]]).sum(1)
        cols_list.append(cos)
    ea=np.concatenate(cols_list,axis=1).astype(np.float32)
    ei=torch.tensor(np.stack([s2,d2]),dtype=torch.long); ea=torch.tensor(ea)
    # self-loops: rec=1, flags=0, is_self=1, rel_count=0, cosine=1
    D=ea.shape[1]; self_ea=torch.zeros((N,D)); self_ea[:,0:3]=1.0; self_ea[:,8]=1.0
    if USE_COSINE: self_ea[:,10]=1.0
    loops=torch.arange(N); ei=torch.cat([ei,torch.stack([loops,loops])],1); ea=torch.cat([ea,self_ea],0)
    print(f"  TOTAL: {ei.shape[1]:,} edges (incl self-loops), edge_dim={ea.shape[1]}")
    return ei, ea

# cache the built graph so seed re-runs / tweaks skip the kNN+coalesce rebuild
_tag = f"rel{'-'.join(IDENTITY_COLS)}_sim{int(USE_SIMILARITY_EDGES)}_k{K_SIM}_ept{EDGE_PER_TRANS}_cos{int(USE_COSINE)}_{SIM_BACKEND}_fulltrain{int(FULL_TRAIN_LABEL_REVEAL)}"
_ei_p, _ea_p = OUT/f"edge_index_{_tag}.npy", OUT/f"edge_attr_{_tag}.npy"
if _ei_p.exists() and _ea_p.exists():
    edge_index = torch.from_numpy(np.load(_ei_p)); edge_attr = torch.from_numpy(np.load(_ea_p))
    print(f"loaded cached graph: {edge_index.shape[1]:,} edges, edge_dim={edge_attr.shape[1]}")
else:
    t0=time.time(); edge_index, edge_attr = build_edges(); print(f"edges built in {time.time()-t0:.0f}s")
    np.save(_ei_p, edge_index.numpy()); np.save(_ea_p, edge_attr.numpy())
EDGE_DIM = edge_attr.shape[1]
if ADD_SIM_REF_FRAUD:
    # fraud rate among each node's PAST REFERENCE similarity neighbors (leak-free:
    # only reference labels, sim edges are causal time<current & cross-uid, self excluded, no val/query labels)
    _src=edge_index[0].numpy(); _dst=edge_index[1].numpy(); _issim=edge_attr[:,7].numpy()>0.5
    _knownnode=np.zeros(N,bool); _knownnode[label_known_idx]=True
    _sel=_issim & _knownnode[_src]
    _fsum=np.zeros(N); _fcnt=np.zeros(N)
    np.add.at(_fsum, _dst[_sel], y_all[_src[_sel]].astype(np.float64))
    np.add.at(_fcnt, _dst[_sel], 1.0)
    _base=float(y_all[label_known_idx].mean()) if len(label_known_idx) else 0.0
    _rate=np.where(_fcnt>0, _fsum/np.maximum(_fcnt,1.0), _base).astype(np.float32)
    _extra=np.stack([_rate, _fcnt.astype(np.float32)],axis=1)   # [sim_ref_fraud_rate, sim_ref_count]
    _mu=_extra[train_mask].mean(0); _sd=_extra[train_mask].std(0)+1e-6
    X_scaled=np.concatenate([X_scaled, ((_extra-_mu)/_sd).astype(np.float32)], axis=1)
    print(f'added sim_ref_fraud_rate + sim_ref_count (leak-free) -> node feats now {X_scaled.shape[1]}')

if PRUNE_TOPK:
    _e0 = edge_index.shape[1]
    edge_index, edge_attr = prune_topk_per_dst(edge_index, edge_attr, PRUNE_K)
    print(f"top-{PRUNE_K} prune: {_e0:,} -> {edge_index.shape[1]:,} edges (strongest-K per node)")


  uid: 1,791,803 edges
  similarity edges: DISABLED (internal-only mode)
  TOTAL: 2,382,343 edges (incl self-loops), edge_dim=11
edges built in 10s
top-15 prune: 2,382,343 -> 2,382,343 edges (strongest-K per node)


In [43]:
# ===== Edge-aware GTAN (edge_dim from edge_attr) =====
def _make_y_input(y, known_idx, n_classes):
    yi=torch.full((y.shape[0],), n_classes, dtype=torch.long); yi[known_idx]=y[known_idx]; return yi

class GTANEdge(nn.Module):
    def __init__(self, in_feats, hidden=32, heads=4, layers=2, n_classes=2, drop=0.2, edge_dim=11):
        super().__init__(); width=hidden*heads; self.n_classes=n_classes
        self.label_emb=nn.Embedding(n_classes+1, in_feats, padding_idx=n_classes)
        self.feat_lin=nn.Linear(in_feats,width); self.label_lin=nn.Linear(in_feats,width)
        self.label_proc=nn.Sequential(nn.BatchNorm1d(width),nn.PReLU(),nn.Dropout(drop),nn.Linear(width,in_feats))
        self.input_drop=nn.Dropout(drop)
        self.convs=nn.ModuleList(); self.norms=nn.ModuleList(); dim=in_feats
        for _ in range(layers):
            self.convs.append(TransformerConv(dim,hidden,heads=heads,concat=True,beta=True,dropout=drop,edge_dim=edge_dim))
            self.norms.append(nn.LayerNorm(width)); dim=width
        self.act=nn.PReLU(); self.drop=nn.Dropout(drop); self.emb_dim=dim
        self.head=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.PReLU(),nn.Dropout(drop),nn.Linear(dim,n_classes))
    def forward(self,x,ei,yin,ea=None):
        le=self.input_drop(self.label_emb(yin)); h=x+self.label_proc(self.feat_lin(x)+self.label_lin(le))
        for conv,norm in zip(self.convs,self.norms): h=self.drop(self.act(norm(conv(h,ei,edge_attr=ea))))
        return self.head(h), h


In [44]:
# ===== Train GTAN with full train labels available to neighbours; score validation =====
def _make_data(y_input):
    data = Data(x=torch.tensor(X_scaled).float(), edge_index=edge_index, y=torch.tensor(y_all, dtype=torch.long))
    data.edge_attr = edge_attr.float()
    data.yik = y_input.clone().long()
    if USE_WEIGHTED_SAMPLING:
        _cw = edge_attr[:,10].clamp(min=0,max=1) if edge_attr.shape[1]>=11 else torch.zeros(edge_attr.shape[0])
        data.edge_weight = (edge_attr[:,9] + edge_attr[:,1] + _cw + 1e-3).float()
    if USE_TEMPORAL_SAMPLING:
        data.time = torch.as_tensor(times, dtype=torch.long)
    return data

def _mk_loader(data, seeds, shuffle, batch_size=GTAN_BATCH):
    kw = {}
    if USE_WEIGHTED_SAMPLING: kw["weight_attr"] = "edge_weight"
    if USE_TEMPORAL_SAMPLING: kw["time_attr"] = "time"
    try:
        return NeighborLoader(data, num_neighbors=list(GTAN_NEIGH), input_nodes=torch.tensor(seeds),
                              batch_size=batch_size, shuffle=shuffle, **kw)
    except Exception as _ex:
        print(f"  [loader] {type(_ex).__name__} with {list(kw)} -> falling back to uniform sampling")
        return NeighborLoader(data, num_neighbors=list(GTAN_NEIGH), input_nodes=torch.tensor(seeds),
                              batch_size=batch_size, shuffle=shuffle)

def _mask_seed_labels(y_input_batch, batch_size):
    # NeighborLoader puts seed/input nodes first. Masking them prevents direct
    # own-label shortcut while leaving sampled train neighbours labelled.
    yb = y_input_batch.clone()
    yb[:batch_size] = 2
    return yb

def run_gtan(seed, return_state=False):
    torch.manual_seed(seed); np.random.seed(seed)
    x = torch.tensor(X_scaled)
    y = torch.tensor(y_all, dtype=torch.long)
    known_tensor = torch.tensor(label_known_idx, dtype=torch.long)
    yik = _make_y_input(y, known_tensor, 2)

    npos = float((y[torch.tensor(query_idx)] == 1).sum().clamp(min=1))
    nneg = float((y[torch.tensor(query_idx)] == 0).sum().clamp(min=1))
    weight = torch.tensor([1.0, float(np.sqrt(nneg/npos))], device=device)

    model = GTANEdge(x.shape[1], GTAN_HIDDEN, GTAN_HEADS, GTAN_LAYERS, 2, GTAN_DROP, EDGE_DIM).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=GTAN_LR, weight_decay=GTAN_WD)
    data = _make_data(yik)
    tl = _mk_loader(data, query_idx, True)
    vl = _mk_loader(data, val_idx, False)

    best, bestp, best_state, bad = -1.0, None, None, 0
    for ep in range(1, GTAN_EPOCHS+1):
        t_epoch = time.time()
        model.train(); tot = nb = 0
        for b in tl:
            bs = b.batch_size; b = b.to(device)
            opt.zero_grad()
            yin = _mask_seed_labels(b.yik, bs)
            logits, _ = model(b.x, b.edge_index, yin, b.edge_attr)
            loss = F.cross_entropy(logits[:bs], b.y[:bs], weight=weight)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            tot += float(loss); nb += 1

        model.eval(); vp = np.zeros(N, np.float32)
        with torch.no_grad():
            for b in vl:
                bs = b.batch_size; b = b.to(device)
                # validation labels are unknown in yik; train neighbours keep labels.
                logits, _ = model(b.x, b.edge_index, b.yik, b.edge_attr)
                vp[b.n_id[:bs].cpu().numpy()] = F.softmax(logits[:bs], 1)[:,1].cpu().numpy()

        au = roc_auc_score(y_all[val_idx], vp[val_idx])
        print(f"   ep {ep:2d} loss={tot/max(nb,1):.4f} val_auc={au:.4f} ({time.time()-t_epoch:.1f}s)")
        if au > best:
            best = au
            bestp = vp[val_idx].copy()
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1
            if bad >= EARLY_STOP_PATIENCE:
                print(f"   early stop at ep {ep}")
                break

    print(f">>> seed {seed}: best val AUC={best:.4f}")
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    if return_state:
        return best, bestp, best_state
    return best, bestp

def _predict_and_embed_from_state(state_dict, known_idx, target_idx, batch_size=GTAN_BATCH):
    y = torch.tensor(y_all, dtype=torch.long)
    yik = _make_y_input(y, torch.tensor(known_idx, dtype=torch.long), 2)
    model = GTANEdge(X_scaled.shape[1], GTAN_HIDDEN, GTAN_HEADS, GTAN_LAYERS, 2, GTAN_DROP, EDGE_DIM).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    data = _make_data(yik)
    loader = _mk_loader(data, target_idx, False, batch_size=batch_size)
    pred = np.zeros(N, np.float32)
    emb = np.zeros((N, model.emb_dim), np.float32)
    with torch.no_grad():
        for b in loader:
            bs = b.batch_size; b = b.to(device)
            logits, h = model(b.x, b.edge_index, b.yik, b.edge_attr)
            ids = b.n_id[:bs].cpu().numpy()
            pred[ids] = F.softmax(logits[:bs], 1)[:,1].cpu().numpy()
            emb[ids] = h[:bs].cpu().numpy()
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return pred, emb

def extract_downstream_embeddings(state_dict):
    # Validation gets the intended upper-bound view: all train labels known.
    val_pred, val_emb = _predict_and_embed_from_state(state_dict, train_idx, val_idx)
    gtan_auc = roc_auc_score(y_all[val_idx], val_pred[val_idx])
    print(f"GTAN embedding model validation AUC = {gtan_auc:.4f}")

    full_emb = np.zeros_like(val_emb)
    if USE_OOF_TRAIN_EMBEDDINGS:
        # Train embeddings are extracted out-of-fold so a train row's own label does
        # not become a feature for the downstream CNN/LSTM. This is expensive.
        full_emb[val_idx] = val_emb[val_idx]
        rng = np.random.default_rng(REF_SEED)
        shuffled = rng.permutation(train_idx)
        for fold, held in enumerate(np.array_split(shuffled, TRAIN_EMB_OOF_FOLDS), start=1):
            known = np.setdiff1d(train_idx, held, assume_unique=False)
            _, held_emb = _predict_and_embed_from_state(state_dict, known, held)
            full_emb[held] = held_emb[held]
            print(f"  train embedding OOF fold {fold}/{TRAIN_EMB_OOF_FOLDS}: held={len(held):,}")
    else:
        # Fast upper-bound extraction: one pass over train+val with all train labels known.
        # This is not own-label-safe for downstream train embeddings, but is useful when
        # the goal is to test whether full-label GTAN has any headroom at all.
        print("FAST embedding extraction: one pass over train+val; train embeddings may include own labels.")
        target = np.concatenate([train_idx, val_idx])
        _, all_emb = _predict_and_embed_from_state(state_dict, train_idx, target)
        full_emb[target] = all_emb[target]

    mu = full_emb[train_idx].mean(0)
    sd = full_emb[train_idx].std(0) + 1e-6
    full_emb = ((full_emb - mu) / sd).astype(np.float32)
    return full_emb, val_pred

seed_aucs=[]; seed_preds=[]; seed_states=[]
for s in range(N_SEEDS):
    print(f"\n-- GTAN seed {SEED+s} --")
    a, p, st = run_gtan(SEED+s, return_state=True)
    seed_aucs.append(a); seed_preds.append(p); seed_states.append(st)

avg = np.mean(seed_preds, axis=0)
best_seed_pos = int(np.argmax(seed_aucs))
BEST_GTAN_STATE = seed_states[best_seed_pos]
print(f"\n=== GTAN-alone uid-disjoint: full train-label validation protocol ===")
print(f"per-seed val AUC: {[round(a,4) for a in seed_aucs]}")
print(f"seed-averaged val AUC = {roc_auc_score(y_all[val_idx], avg):.4f}")
print(f"seed-averaged val AP  = {average_precision_score(y_all[val_idx], avg):.4f}")
print(f"embedding seed selected: {SEED+best_seed_pos} (AUC={seed_aucs[best_seed_pos]:.4f})")

GTAN_EMB, GTAN_VAL_PRED = extract_downstream_embeddings(BEST_GTAN_STATE)



-- GTAN seed 42 --
   ep  1 loss=0.3551 val_auc=0.8611
   ep  2 loss=0.2886 val_auc=0.8695
   ep  3 loss=0.2685 val_auc=0.8691
   ep  4 loss=0.2539 val_auc=0.8604
   ep  5 loss=0.2404 val_auc=0.8645
   ep  6 loss=0.2323 val_auc=0.8549
   early stop at ep 6
>>> seed 42: best val AUC=0.8695

-- GTAN seed 43 --
   ep  1 loss=0.3680 val_auc=0.8650
   ep  2 loss=0.2888 val_auc=0.8720
   ep  3 loss=0.2718 val_auc=0.8744
   ep  4 loss=0.2569 val_auc=0.8662
   ep  5 loss=0.2450 val_auc=0.8620
   ep  6 loss=0.2341 val_auc=0.8383
   ep  7 loss=0.2245 val_auc=0.8414
   early stop at ep 7
>>> seed 43: best val AUC=0.8744

-- GTAN seed 44 --
   ep  1 loss=0.3631 val_auc=0.8665
   ep  2 loss=0.2893 val_auc=0.8681
   ep  3 loss=0.2688 val_auc=0.8735
   ep  4 loss=0.2541 val_auc=0.8617
   ep  5 loss=0.2406 val_auc=0.8554
   ep  6 loss=0.2312 val_auc=0.8435
   ep  7 loss=0.2222 val_auc=0.8495
   early stop at ep 7
>>> seed 44: best val AUC=0.8735

=== GTAN-alone uid-disjoint ===
per-seed val AUC: [np.

## CNN/LSTM hybrids with GTAN embeddings

In [ ]:
# ===== Load UID-window sequences and align them to the dataframe row positions =====
X_seq_aligned = np.load(f"{SPLIT_DATA_DIR}/X_train_seq.npy")
L_aligned = np.load(f"{SPLIT_DATA_DIR}/L_train.npy")
y_aligned = np.load(f"{SPLIT_DATA_DIR}/y_aligned.npy").astype(np.int64)
train_order = np.load(f"{SPLIT_DATA_DIR}/train_order.npy").astype(np.int64)

row_to_pos = pd.Series(np.arange(N), index=df.index.astype(np.int64))
aligned_pos = row_to_pos.loc[train_order].to_numpy()

X_seq = np.empty_like(X_seq_aligned)
L_all_seq = np.empty_like(L_aligned)
y_seq = np.empty_like(y_aligned)
X_seq[aligned_pos] = X_seq_aligned
L_all_seq[aligned_pos] = L_aligned
y_seq[aligned_pos] = y_aligned

assert np.array_equal(y_seq, y_all), "Sequence labels do not align with dataframe/y_all rows."
N_seq, T, NF = X_seq.shape
print("X_seq aligned:", X_seq.shape, "| features:", NF, "| train rows:", len(train_idx), "| val rows:", len(val_idx))


In [ ]:
# ===== Window dataset + UID-disjoint hybrid runner =====
class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = X
        self.lengths = torch.from_numpy(lengths.astype("int64"))
        self.y = None if y is None else torch.from_numpy(y.astype("float32"))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        x = torch.from_numpy(np.asarray(self.X[i])).float()
        return (x, self.lengths[i]) if self.y is None else (x, self.lengths[i], self.y[i])

def make_loader(X, lengths, y, batch_size, shuffle):
    return DataLoader(WindowDataset(X, lengths, y), batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=False, drop_last=False)

def augment_rows(rows, embedding):
    row_sequences = np.asarray(X_seq[rows])
    n_rows = len(rows)
    emb_dim = embedding.shape[1]
    repeated_embedding = np.zeros((n_rows, T, emb_dim), dtype=np.float16)
    lengths = L_all_seq[rows].astype(int)
    for position, global_idx in enumerate(rows):
        repeated_embedding[position, T-lengths[position]:, :] = embedding[global_idx]
    return np.concatenate([row_sequences, repeated_embedding], axis=2)

def run_uid_disjoint_hybrid(tag):
    oof = np.full(N, np.nan, dtype=np.float32)
    Xtr = augment_rows(train_idx, GTAN_EMB)
    Xva = augment_rows(val_idx, GTAN_EMB)
    ytr, yva = y_all[train_idx].astype("float32"), y_all[val_idx].astype("float32")
    Ltr, Lva = L_all_seq[train_idx], L_all_seq[val_idx]
    nf = Xtr.shape[2]
    seed_preds = []
    for s in range(N_SEEDS):
        print(f"\n-- {tag} seed {SEED+s} --")
        vp, va, model = train_one_fold(
            Xtr, Ltr, ytr, Xva, Lva, yva, n_features=nf,
            epochs=EPOCHS, batch=BATCH, lr=LR, weight_decay=WEIGHT_DECAY,
            device=device, early_stop_patience=EARLY_STOP_PATIENCE,
            grad_clip=GRAD_CLIP, seed=SEED+s,
        )
        seed_preds.append(vp)
        del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    fold_pred = np.mean(seed_preds, axis=0)
    oof[val_idx] = fold_pred
    auc = roc_auc_score(yva, fold_pred)
    ap = average_precision_score(yva, fold_pred)
    print(f"=== {tag} UID-disjoint AUC = {auc:.4f} | AP = {ap:.4f}\n")
    del Xtr, Xva; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return oof


### CNN + GTAN

In [ ]:
"""
Layout B v2 — Conv1D + ResNet1D + STACKED Self-Attention + CLS + Mean/Max Pool + Static Tower.

Upgrades over fraud_cnn_resnet_attention.py:
  (1) Masked Mean + Max pool concatenated     → captures average + spike behavior
  (2) Learnable CLS token                     → transformer-style classification readout
  (3) Static-row tower (last transaction MLP) → recovers XGBoost-style intra-row signal
  (3) Stack of N transformer encoder blocks   → multi-hop attention reasoning

Same forward signature as v1:
    forward(x, lengths) -> logits of shape (B, 1)

Pipeline:
    (B, T, F)
        ├── transpose ───────────────────────► (B, F, T)
        ├── Conv1D stem (F → C, k=3) ────────► (B, C, T)
        ├── ResBlock1D × N_RESBLOCKS ────────► (B, C, T)
        ├── transpose + positional emb ──────► (B, T, C)
        ├── [optionally prepend CLS token] ──► (B, T+1, C)
        ├── TransformerBlock × N_ATTN_LAYERS ► (B, T+1, C)
        ├── pooling:
        │     - mean over real timesteps     → (B, C)
        │     - max  over real timesteps     → (B, C)        [optional]
        │     - CLS readout                  → (B, C)        [optional]
        │     - static MLP on last row       → (B, S)        [optional]
        ├── concatenate the parts            ► (B, pool_dim)
        └── MLP head                         ► (B, 1)
"""

import torch
from torch import nn


# =============================================================================
# 1D Residual Block — shape preserving (unchanged from v1)
# =============================================================================
class ResBlock1D(nn.Module):
    def __init__(self, c: int, k: int = 3, drop: float = 0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn1   = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn2   = nn.BatchNorm1d(c)
        self.drop  = nn.Dropout(drop)
        self.act   = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x                                   # SKIP BRANCH
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = out + identity                           # SKIP CONNECTION
        return self.act(out)


# =============================================================================
# Transformer encoder block — one self-attention + FF, both with residual+LN
# =============================================================================
class TransformerBlock(nn.Module):
    def __init__(self, c: int, n_heads: int, drop: float = 0.2):
        super().__init__()
        self.attn  = nn.MultiheadAttention(
            embed_dim=c, num_heads=n_heads, dropout=drop, batch_first=True,
        )
        self.norm1 = nn.LayerNorm(c)
        self.ff    = nn.Sequential(
            nn.Linear(c, c * 2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(c * 2, c),
        )
        self.norm2 = nn.LayerNorm(c)

    def forward(self, h: torch.Tensor, key_padding_mask: torch.Tensor) -> torch.Tensor:
        a, _ = self.attn(h, h, h,
                         key_padding_mask=key_padding_mask,
                         need_weights=False)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ff(h))
        return h


# =============================================================================
# Full v2 model
# =============================================================================
class FraudCNNResAttnV2(nn.Module):
    """
    Args
    ----
    n_features        : number of features per timestep (e.g. 244)
    window            : sequence length T (e.g. 20)
    c_hidden          : channel width of Conv stem / ResNet / attention
    n_resblocks       : number of ResBlock1D in the CNN stack
    n_attn_layers     : number of TransformerBlock layers (was 1 in v1)        ◄── NEW
    n_heads           : self-attention heads (must divide c_hidden)
    drop              : dropout used in ResBlocks, attention, FF, head, static tower
    static_hidden     : hidden width of the static-row tower
    use_cls           : prepend a learnable CLS token, use its embedding for readout ◄── NEW
    use_max_pool      : concat masked-max pool with mean pool                      ◄── NEW
    use_static_tower  : run the last raw row through an MLP and concat it          ◄── NEW
    output_dim        : final logit dim (keep =1 for BCEWithLogitsLoss)
    """
    def __init__(self,
                 n_features: int,
                 window: int = 20,
                 c_hidden: int = 128,
                 n_resblocks: int = 3,
                 n_attn_layers: int = 2,
                 n_heads: int = 4,
                 drop: float = 0.2,
                 static_hidden: int = 64,
                 use_cls: bool = True,
                 use_max_pool: bool = True,
                 use_static_tower: bool = True,
                 output_dim: int = 1):
        super().__init__()
        assert c_hidden % n_heads == 0, "c_hidden must be divisible by n_heads"

        self.window           = window
        self.use_cls          = use_cls
        self.use_max_pool     = use_max_pool
        self.use_static_tower = use_static_tower

        # ------- Stage 1 — Conv1D stem -------
        self.stem = nn.Sequential(
            nn.Conv1d(n_features, c_hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(c_hidden),
            nn.ReLU(),
        )

        # ------- Stage 2 — ResNet1D stack -------
        self.resblocks = nn.Sequential(*[
            ResBlock1D(c_hidden, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])

        # ------- Stage 3 — positional embedding (over the WINDOW positions only;
        #                  the CLS token is added on top after this)
        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden))
        nn.init.trunc_normal_(self.pos, std=0.02)

        # ------- (Optional) CLS token -------
        if use_cls:
            self.cls = nn.Parameter(torch.zeros(1, 1, c_hidden))
            nn.init.trunc_normal_(self.cls, std=0.02)

        # ------- Stage 4 — STACK of Transformer blocks -------
        self.attn_blocks = nn.ModuleList([
            TransformerBlock(c_hidden, n_heads, drop=drop)
            for _ in range(n_attn_layers)
        ])

        # ------- (Optional) Static-row tower -------
        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
                nn.Linear(256, static_hidden), nn.ReLU(),
            )

        # ------- Compute pool dim from the toggles -------
        pool_dim = c_hidden                       # mean is always present
        if use_max_pool:    pool_dim += c_hidden
        if use_cls:         pool_dim += c_hidden
        if use_static_tower:pool_dim += static_hidden

        # ------- Stage 5 — Head -------
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(pool_dim, 64), nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, output_dim),
        )

    # ----------------------------------------------------------------------
    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)  left-padded     lengths: (B,) real-step count
        B, T, F = x.shape

        # Keep last (most-recent) raw row aside for the static tower.
        # Left-padding writes to the RIGHT end, so x[:, -1, :] is always real.
        last_row = x[:, -1, :]                           # (B, F)

        # ----- CNN path (B, T, F) → (B, T, C) -----
        x_t = x.transpose(1, 2)                          # (B, F, T)
        h   = self.stem(x_t)                             # (B, C, T)
        h   = self.resblocks(h)                          # (B, C, T)
        h   = h.transpose(1, 2)                          # (B, T, C)
        h   = h + self.pos                               # positional encoding (over T only)

        # ----- Build real-position mask from `lengths` -----
        idx          = torch.arange(T, device=h.device).unsqueeze(0)   # (1, T)
        pos_from_end = T - 1 - idx                                     # (1, T)
        real         = pos_from_end < lengths.unsqueeze(1)             # (B, T) — True = real

        # ----- (Optional) Prepend CLS token -----
        if self.use_cls:
            cls = self.cls.expand(B, -1, -1)             # (B, 1, C)
            h   = torch.cat([cls, h], dim=1)             # (B, T+1, C)
            # CLS is always "real" so attention won't mask it
            cls_real = torch.ones(B, 1, dtype=torch.bool, device=h.device)
            real_full = torch.cat([cls_real, real], dim=1)              # (B, T+1)
            key_padding_mask = ~real_full
        else:
            key_padding_mask = ~real

        # ----- Stacked Transformer blocks -----
        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)

        # ----- Split CLS embedding from the time tokens -----
        if self.use_cls:
            cls_out = h[:, 0, :]                         # (B, C)
            seq_h   = h[:, 1:, :]                        # (B, T, C)
        else:
            seq_h = h                                    # (B, T, C)

        # ----- Pool over REAL timesteps only -----
        mask_f = real.unsqueeze(-1).float()              # (B, T, 1)
        cnt    = mask_f.sum(dim=1).clamp(min=1.0)        # (B, 1)

        pooled_parts = [(seq_h * mask_f).sum(dim=1) / cnt]   # mean

        if self.use_max_pool:
            # mask out padded positions with -inf so they never win the max
            seq_h_for_max = seq_h.masked_fill(~real.unsqueeze(-1), float('-inf'))
            pooled_parts.append(seq_h_for_max.max(dim=1).values)

        if self.use_cls:
            pooled_parts.append(cls_out)

        if self.use_static_tower:
            pooled_parts.append(self.static_mlp(last_row))

        pooled = torch.cat(pooled_parts, dim=1)          # (B, pool_dim)
        return self.head(pooled)                         # (B, 1)


class FraudCNNResAttnV2PerStep(nn.Module):
    """
    Per-timestep CNN/ResNet/Attention model.

    Input : X shape (B, T, F), left-padded
            lengths shape (B,)
    Output: logits shape (B, T)

    Train with masked BCE:
        loss = BCE(logits[M], Y[M])
    """
    def __init__(self,
                 n_features: int,
                 window: int = 20,
                 c_hidden: int = 128,
                 n_resblocks: int = 3,
                 n_attn_layers: int = 2,
                 n_heads: int = 4,
                 drop: float = 0.2):
        super().__init__()

        assert c_hidden % n_heads == 0

        self.window = window

        self.stem = nn.Sequential(
            nn.Conv1d(n_features, c_hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(c_hidden),
            nn.ReLU(),
        )

        self.resblocks = nn.Sequential(*[
            ResBlock1D(c_hidden, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])

        self.pos = nn.Parameter(torch.zeros(1, window, c_hidden))
        nn.init.trunc_normal_(self.pos, std=0.02)

        self.attn_blocks = nn.ModuleList([
            TransformerBlock(c_hidden, n_heads, drop=drop)
            for _ in range(n_attn_layers)
        ])

        # One logit per timestep
        self.token_head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(c_hidden, 64),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        # x: (B, T, F), LEFT-padded
        B, T, _ = x.shape

        h = x.transpose(1, 2)        # (B, F, T)
        h = self.stem(h)             # (B, C, T)
        h = self.resblocks(h)        # (B, C, T)
        h = h.transpose(1, 2)        # (B, T, C)

        h = h + self.pos[:, :T, :]

        # left-padding mask: real positions are at the RIGHT end
        idx = torch.arange(T, device=x.device).unsqueeze(0)
        real = (T - 1 - idx) < lengths.to(x.device).unsqueeze(1)
        key_padding_mask = ~real

        for blk in self.attn_blocks:
            h = blk(h, key_padding_mask)

        logits = self.token_head(h).squeeze(-1)   # (B, T)
        return logits

In [ ]:
def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudCNNResAttnV2(
        n_features,
        c_hidden=CNN_CHANNELS,
        n_resblocks=N_RESBLOCKS,
        n_attn_layers=N_ATTN_LAYERS,
        n_heads=N_HEADS,
        drop=ATTN_DROPOUT,
        static_hidden=STATIC_HIDDEN,
        use_cls=USE_CLS,
        use_max_pool=USE_MAX_POOL,
        use_static_tower=USE_STATIC_TOWER,
    ).to(device) # CNN_ResNet_Attention v2
    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb = xb.to(device); lb = lb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            logits = logits = model(xb, lb).reshape(-1) 
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb = xb.to(device); lb = lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb).reshape(-1)).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds; bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_preds, best_auc, model

In [ ]:
# CNN+ResNet+Attention + GTAN embeddings only.
# Baseline is intentionally not recomputed in this notebook.
if RUN_HYBRIDS:
    cnn_gtan = run_uid_disjoint_hybrid("CNN + GTAN full-train-label")
    print(f"Recorded CNN baseline UID-disjoint AUC = {CNN_BASELINE_UID_DISJOINT_AUC:.4f}")
    v = ~np.isnan(cnn_gtan)
    print(f"CNN + GTAN UID-disjoint OOF AUC     = {roc_auc_score(y_all[v], cnn_gtan[v]):.4f}")
else:
    cnn_gtan = np.full(N, np.nan, dtype=np.float32)
    print("RUN_HYBRIDS=False: skipped CNN + GTAN")


### LSTM + GTAN

In [ ]:
class FraudLSTM(nn.Module):
    def __init__(self, n_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 drop_prob=DROPOUT, output_dim=1,
                 use_static_tower=USE_STATIC_TOWER,
                 bidirectional=True):                     # NEW
        super().__init__()
        self.use_static_tower = use_static_tower
        self.bidirectional   = bidirectional             # NEW

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
            bidirectional=bidirectional,                 # NEW
        )

        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)   # NEW
        seq_out_dim  = 2 * lstm_out_dim                            # mean + max

        if use_static_tower:
            self.static_mlp = nn.Sequential(
                nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(256, 128),        nn.ReLU(), nn.Dropout(drop_prob),
                nn.Linear(128, 64),         nn.ReLU(),
            )
            combined_dim = seq_out_dim + 64
        else:
            combined_dim = seq_out_dim

        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, output_dim),
        )

    def forward(self, x, lengths):
        # x: (B, T, F)  left-padded; lengths: (B,) real-step counts
        B, T, _ = x.shape
        lstm_out, _ = self.lstm(x)                                  # (B, T, H)

        # build a mask for the real (right-aligned) positions
        idx = torch.arange(T, device=x.device).unsqueeze(0)         # (1, T)
        pos_from_end = T - 1 - idx                                  # 0..T-1
        mask = pos_from_end < lengths.to(x.device).unsqueeze(1)     # (B, T)
        mask_f = mask.unsqueeze(-1).float()

        # masked mean
        sum_  = (lstm_out * mask_f).sum(dim=1)
        cnt   = mask_f.sum(dim=1).clamp(min=1.0)
        mean_pool = sum_ / cnt

        # masked max (pads → very negative)
        neg_inf  = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values

        seq_vec = torch.cat([mean_pool, max_pool], dim=1)

        if self.use_static_tower:
            current  = x[:, -1, :]                                  # last real step
            stat_vec = self.static_mlp(current)
            feat = torch.cat([seq_vec, stat_vec], dim=1)
        else:
            feat = seq_vec

        return self.head(feat).squeeze(-1)

In [ ]:
def train_one_fold(X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudLSTM(n_features).to(device)

    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum()); neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()                            # plain BCE → better AUC

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb = xb.to(device); lb = lb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb, lb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb = xb.to(device); lb = lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')
        if val_auc > best_auc:
            best_auc = val_auc; best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds; bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_preds, best_auc, model

In [ ]:
# LSTM + GTAN embeddings only.
# Baseline is intentionally not recomputed in this notebook.
if RUN_HYBRIDS:
    lstm_gtan = run_uid_disjoint_hybrid("LSTM + GTAN full-train-label")
    print(f"Recorded LSTM baseline UID-disjoint AUC = {LSTM_BASELINE_UID_DISJOINT_AUC:.4f}")
    v = ~np.isnan(lstm_gtan)
    print(f"LSTM + GTAN UID-disjoint OOF AUC     = {roc_auc_score(y_all[v], lstm_gtan[v]):.4f}")
else:
    lstm_gtan = np.full(N, np.nan, dtype=np.float32)
    print("RUN_HYBRIDS=False: skipped LSTM + GTAN")


## Summary

In [ ]:
# ===== Summary =====
print("=" * 82)
print("UID-DISJOINT FULL TRAIN-LABEL GTAN SUMMARY")
print(f"GTAN seed-averaged validation AUC = {roc_auc_score(y_all[val_idx], avg):.4f}")
print(f"GTAN seed-averaged validation AP  = {average_precision_score(y_all[val_idx], avg):.4f}")

for name, pred, base in [
    ("CNN+GTAN", cnn_gtan, CNN_BASELINE_UID_DISJOINT_AUC),
    ("LSTM+GTAN", lstm_gtan, LSTM_BASELINE_UID_DISJOINT_AUC),
]:
    valid = ~np.isnan(pred)
    auc = roc_auc_score(y_all[valid], pred[valid])
    print(f"{name:10s} AUC = {auc:.4f} | recorded baseline = {base:.4f} | change = {auc-base:+.4f}")
print("=" * 82)
